In [ ]:
#use "../lib/nb.ml";;
run_notebook "Types.ipynb"

# Minimizing a Finite State Machine with a Table

In this notebook we implement the algorithm from the section *Minimization of Finite State Machines* of the
lecture notes.  This algorithm minimizes a <span style="font-variant:small-caps;">Dfa</span> by identifying states that are *equivalent*.  The notebook
`07-Minimize.ipynb` implements the same algorithm, but it only returns the minimized <span style="font-variant:small-caps;">Dfa</span>.  In contrast, this
notebook also computes the tables that are used in the lecture notes in order to find the pairs of *separable*
states, and displays them.

## Some Type Definitions

* Characters have the OCaml type `char`.
* The type parameter `'q` is the type of the states that make up the <span style="font-variant:small-caps;">Dfa</span>.

`'q trans_rel_det` is the type that describes the transition function $\delta$.  It is defined in the notebook `Types.ipynb`.

In [ ]:
#show_type trans_rel_det

`'q dfa` is the type that describes a <span style="font-variant:small-caps;">Dfa</span>.

In [ ]:
#show_type dfa

`'q pair` is a pair of states.

In [ ]:
type 'q pair = 'q * 'q

A `separation_table` stores the pairs of states that are known to be *separable*.  It corresponds to the
tables shown in the lecture notes:  If the pair $(p, q)$ has been recognized as separable in step $k$ of the
algorithm, then `table` maps the pair $(p, q)$ to $k$.  Pairs that are not known to be separable do not occur in the table.
As separability is symmetric, the pair $(q, p)$ is in the table whenever the pair $(p, q)$ is.

In [ ]:
type 'q separation_table = ('q pair, int) Dict.t

The states of the minimized <span style="font-variant:small-caps;">Dfa</span> will be  sets of states of the original <span style="font-variant:small-caps;">Dfa</span>.  These states will contain those states that are *equivalent*.

In [ ]:
type 'q set_state = 'q FSet.t

In [ ]:
type 'q trans_rel_set = 'q set_state trans_rel_det

In [ ]:
type 'q min_dfa = 'q set_state dfa

## Computing the Table

The function `cart_prod a b` computes the *Cartesian product* $A \times B$ of the sets $A$ and $B$ where $A \times B$ is defined as follows:
$$ A \times B := \{ (x, y) \mid x \in A \wedge y \in B \}. $$
The type variables `'s` and `'t` denote the types of the elements of the sets.

**Function `cart_prod a b`**
- *Input:* `a` and `b` are sets.
- *Output:* The Cartesian product $A \times B$.

In [ ]:
let cart_prod (a : 's FSet.t) (b : 't FSet.t) : ('s * 't) FSet.t =
  let open FSet in
  let* x = a in
  let* y = b in
  return (x, y)

The function `reachable q0 sigma delta` takes three arguments:
* `q0` is the start state of a <span style="font-variant:small-caps;">Dfa</span>,
* `sigma`  is the alphabet.
* `delta`  is the transition function.  The transition function is assumed to be *complete*. `delta` is represented as a dictionary.

It returns the set of all states that can be reached from the start state `q0` by reading strings of characters from `sigma`.
The lecture notes assume that all states of the <span style="font-variant:small-caps;">Dfa</span> are reachable.  Therefore, the states that are not reachable
are removed before the table is computed.

The result is computed via a *fixed point* computation.

**Function `reachable q0 sigma delta`**
- *Input:* `q0` is the start state of a complete <span style="font-variant:small-caps;">Dfa</span>, `sigma` is its alphabet, and `delta` is its transition function.
- *Output:* The set of all states that can be reached from `q0`.

In [ ]:
let reachable (q0 : 'q) (sigma : char FSet.t) (delta : 'q trans_rel_det) : 'q FSet.t =
  let rec loop (result : 'q FSet.t) : 'q FSet.t =
    let newStates =
      let open FSet in
      let* p = result in
      let* c = sigma in
      return (Dict.find (p, c) delta)
    in
    if FSet.subset newStates result then result
    else loop (FSet.union result newStates)
  in
  loop (FSet.singleton q0)

The function `separation_table q a sigma delta` takes four arguments:
* `q` is the set of states of the <span style="font-variant:small-caps;">Dfa</span>.
* `a` is the set of all accepting states,
* `sigma`  is the alphabet.
* `delta` is the transition function.

  `delta` is represented as a dictionary.

The function computes the table of all pairs of separable states that is described in the lecture notes.
To this end, it computes the sequence of sets $V_0 \subseteq V_1 \subseteq \cdots \subseteq V_K$ of pairs of
separable states that is defined by induction:
1. Initially, all accepting states are separated from the non-accepting states:
   $$ V_0 := \bigl(A \times (Q \backslash A)\bigr) \cup \bigl((Q \backslash A) \times A\bigr). $$
2. If $q_1$ and $q_2$ are two states and there is a character $c \in \Sigma$ such that the states
   $\delta(q_1, c)$ and $\delta(q_2, c)$ are already known to be separable, then $q_1$ and $q_2$ are separable, too:
   $$ V_{k+1} := V_k \cup \bigl\{ (q_1, q_2) \bigm| \exists c \in \Sigma: \bigl(\delta(q_1, c), \delta(q_2, c)\bigr) \in V_k \bigr\}. $$

The computation stops as soon as no new pairs are found, i.e. as soon as $V_{K+1} = V_K$.
As in the lecture notes, the steps are numbered starting with $1$:  The pairs of $V_0$ are entered into the table
with the number $1$ and, for $k \geq 1$, the pairs of $V_k \backslash V_{k-1}$ are entered with the number $k+1$.
Before the step $k + 1$ is performed, the table contains exactly the pairs of $V_{k-1}$.

As in the Python version, the table is extended in a loop.  Since dictionaries are immutable, `table` is a
reference to a dictionary.

**Function `separation_table q a sigma delta`**
- *Input:* `q` is the set of states of a complete <span style="font-variant:small-caps;">Dfa</span>, `a` is its set of accepting states, `sigma` is its alphabet, and `delta` is its transition function.
- *Output:* A table that maps every pair of separable states to the number of the step in which this pair has been recognized as separable.

In [ ]:
let separation_table (q : 'q FSet.t) (a : 'q FSet.t) (sigma : char FSet.t) (delta : 'q trans_rel_det)
    : 'q separation_table =
  let table =
    ref (Dict.of_list (List.map (fun pair -> (pair, 1))
                                (FSet.to_list (FSet.union (cart_prod a (FSet.diff q a)) (cart_prod (FSet.diff q a) a)))))
  in
  let rec loop (k : int) : 'q separation_table =
    let newPairs =
      let open FSet in
      let* q1 = q in
      let* q2 = q in
      let* c  = sigma in
      if not (Dict.mem (q1, q2) !table) && Dict.mem (Dict.find (q1, c) delta, Dict.find (q2, c) delta) !table
      then return (q1, q2)
      else empty
    in
    if FSet.is_empty newPairs then !table
    else begin
      let k = k + 1 in
      FSet.iter (fun pair -> table := Dict.add pair k !table) newPairs;
      loop k
    end
  in
  loop 1

## Displaying the Tables

The tables are displayed with the function `display_html` from the file `../lib/nb.ml`.

In [ ]:
(* display_html is defined in ../lib/nb.ml *)

The function `state_name q` returns the name of the state `q` that is used when the tables and the <span style="font-variant:small-caps;">Dfa</span>s are displayed.
In the examples below, the states are strings, so a state is its own name.
The states of a minimized <span style="font-variant:small-caps;">Dfa</span> are sets of states.  These are displayed as sets of the form
$\{q_1, \cdots, q_n\}$, where the states $q_i$ are sorted by their names.  This is done by the function
`set_name ~name m`, which uses the function `name` to compute the names of the elements of the set `m`.

**Type safety:** The Python function `state_name` accepts arbitrary objects and uses `isinstance` to find out at
runtime whether its argument is a set.  OCaml does not inspect types at runtime.  Instead, the functions that
display tables and <span style="font-variant:small-caps;">Dfa</span>s take the function `name` that computes the
name of a state as a labelled argument `~name`.  For a <span style="font-variant:small-caps;">Dfa</span> with string states, this is
`state_name`, while for the minimized <span style="font-variant:small-caps;">Dfa</span> it is `set_name ~name:state_name`.  If we
pass the wrong function, the compiler reports a type error.

**Function `state_name q`**
- *Input:* `q` is a state, i.e. a string.
- *Output:* The name of `q` as a string.

**Function `set_name ~name m`**
- *Input:* `m` is a set of states and `name` is a function that computes the names of these states.
- *Output:* The name of `m` as a string.

In [ ]:
let state_name (q : string) : string = q

let set_name ~(name : 'q -> string) (m : 'q FSet.t) : string =
  "{" ^ String.concat ", " (List.sort compare (List.map name (FSet.to_list m))) ^ "}"

The function `show_table ~name ?final states table k` returns the table after step `k` as an HTML string.  Its rows
and columns are labelled with the states.  If the pair $(p, q)$ has been recognized as separable in one of the
steps $1$, $\cdots$, $k$, the entry in the row $p$ and the column $q$ is the number of this step.  As every state
is equivalent to itself, the entries on the diagonal are marked with the symbol $\sim$ from the beginning.  All
other entries are empty.  The list `states` determines the order of the rows and columns.

If the flag `final` is `true`, the table is the *final table* that is computed after the last step.  In this
table, every pair $(p, q)$ of states that could not be separated is marked with the symbol $\sim$, as the
states $p$ and $q$ are equivalent.  `final` is an *optional labelled argument*: it is passed as `~final:true`.

**Function `show_table ~name ?final states table k`**
- *Input:* `states` is a list of the states of a complete <span style="font-variant:small-caps;">Dfa</span>, `table` is a separation table for this <span style="font-variant:small-caps;">Dfa</span>, `k` is a step number, `final` is a flag that is `false` by default, and `name` computes the names of the states.
- *Output:* An HTML string that shows the table after step `k`.  The diagonal is marked with $\sim$.  If `final` is `true`, all pairs of states that have not been separated are marked with $\sim$.

In [ ]:
let show_table ~(name : 'q -> string) ?(final : bool = false) (states : 'q list) (table : 'q separation_table) (k : int)
    : string =
  let style = "border: 1px solid; padding: 2px 12px; text-align: center" in
  let th    = Printf.sprintf "<th style=\"%s\">" style in
  let td    = Printf.sprintf "<td style=\"%s\">" style in
  let equiv = "<span style=\"color: rgb(0, 100, 220); font-size: 130%; font-weight: bold\">∼</span>" in
  let html  = Buffer.create 1024 in
  let add s = Buffer.add_string html s in
  add ("<table style=\"border-collapse: collapse\">\n<tr>" ^ th ^ "</th>");
  List.iter (fun q -> add (th ^ name q ^ "</th>")) states;
  add "</tr>\n";
  List.iter
    (fun p ->
      add ("<tr>" ^ th ^ name p ^ "</th>");
      List.iter
        (fun q ->
          match Dict.find_opt (p, q) table with
          | Some n when n <= k -> add (td ^ string_of_int n ^ "</td>")
          | _ -> if p = q || final then add (td ^ equiv ^ "</td>") else add (td ^ "</td>"))
        states;
      add "</tr>\n")
    states;
  add "</table>";
  Buffer.contents html

The function `explain_step ~name states table k sigma delta` explains why the pairs that are entered into the table in
step `k` are separable:
- In step $1$, every accepting state is separated from every non-accepting state.
- If $k > 1$ and the pair $(p, q)$ is entered in step $k$, then there is a character $c$ such that the pair
  $\bigl(\delta(p, c), \delta(q, c)\bigr)$ has been entered in an earlier step.

As the table is symmetric, only those pairs $(p, q)$ are explained where $p$ precedes $q$ in the list `states`.
If there are several characters that separate $p$ and $q$, the smallest of them is shown.

**Function `explain_step ~name states table k sigma delta`**
- *Input:* `states` is a list of the states of a complete <span style="font-variant:small-caps;">Dfa</span>, `table` is a separation table for this <span style="font-variant:small-caps;">Dfa</span>, `k` is a step number, `sigma` is the alphabet, `delta` is the transition function, and `name` computes the names of the states.
- *Output:* A string that explains why the pairs that are entered in step `k` are separable.

`FSet.choose` returns the smallest element of a set, so it corresponds to Python's function `min`.

In [ ]:
let explain_step ~(name : 'q -> string) (states : 'q list) (table : 'q separation_table) (k : int)
    (sigma : char FSet.t) (delta : 'q trans_rel_det) : string =
  if k = 1 then "Step 1: Every accepting state is separable from every non-accepting state."
  else begin
    let result = Buffer.create 256 in
    let add s = Buffer.add_string result s in
    add (Printf.sprintf "Step %d:" k);
    List.iteri
      (fun i p ->
        List.iteri
          (fun j q ->
            if j > i && Dict.find_opt (p, q) table = Some k then begin
              let c = FSet.choose (FSet.filter (fun c -> Dict.get (Dict.find (p, c) delta, Dict.find (q, c) delta) ~default:k table < k) sigma) in
              let p1, q1 = name (Dict.find (p, c) delta), name (Dict.find (q, c) delta) in
              add (Printf.sprintf "\n  %s ≁ %s, because " (name p) (name q));
              add (Printf.sprintf "𝛿(%s, %c) = %s, 𝛿(%s, %c) = %s, and %s ≁ %s." (name p) c p1 (name q) c q1 p1 q1)
            end)
          states)
      states;
    Buffer.contents result
  end

Two states $p$ and $q$ are *equivalent*, written $p \sim q$, iff they are not separable, i.e. iff the pair
$(p, q)$ does not occur in the final table.  The function `equivalence_class q qs table` computes the
*equivalence class* of the state $q$, which is defined as
$$ [q]_\sim := \{ p \in Q \mid p \sim q \}. $$

**Function `equivalence_class q qs table`**
- *Input:* `q` is a state, `qs` is the set of all states, and `table` is the final separation table.
- *Output:* The equivalence class $[q]_\sim$ of `q`.

In [ ]:
let equivalence_class (q : 'q) (qs : 'q FSet.t) (table : 'q separation_table) : 'q set_state =
  FSet.filter (fun p -> not (Dict.mem (p, q) table)) qs

The function `show_tables ~name f` takes a complete <span style="font-variant:small-caps;">Dfa</span> `f`, removes its unreachable states, and computes the
separation table.  Then, for every step $k$ that finds new pairs of separable states, it explains this step
and displays the table after this step.  Finally, it displays the final table, where the pairs of states that
could not be separated are marked as equivalent, and prints the equivalence classes.

The kernel of *Jupyter* sends printed text and displayed tables through different channels, so printed text might
appear after a table that is displayed later.  Therefore, the texts are shown with the function `display_text` from
the file `../lib/nb.ml`, which uses the same channel as `display_html`.

**Function `show_tables ~name f`**
- *Input:* `f` is a complete <span style="font-variant:small-caps;">Dfa</span>.
- *Output:* `()`.  The tables are displayed as a side effect.

In [ ]:
(* display_html is defined in ../lib/nb.ml *)

In [ ]:
let show_tables ~(name : 'q -> string) (f : 'q dfa) : unit =
  let ({ sigma; delta; start = q0; accepting = a; _ } : 'q dfa) = f in
  let q      = reachable q0 sigma delta in
  let states = List.sort (fun p1 p2 -> compare (name p1) (name p2)) (FSet.to_list q) in
  let table  = separation_table q (FSet.inter a q) sigma delta in
  let kk     = Dict.fold (fun _ v m -> max v m) table 1 in
  for k = 1 to kk do
    display_text (explain_step ~name states table k sigma delta);
    display_html (show_table ~name states table k)
  done;
  display_text (Printf.sprintf "Step %d: No new pairs of separable states are found.\n" (kk + 1)
                ^ "Hence, the pairs of states that could not be separated are equivalent:");
  display_html (show_table ~name ~final:true states table kk);
  let classes = FSet.map (fun p -> equivalence_class p q table) q in
  let sorted  = List.sort (fun c1 c2 -> compare (set_name ~name c1) (set_name ~name c2)) (FSet.to_list classes) in
  display_text (String.concat "\n" ("The equivalence classes are:" :: List.map (fun c -> "  " ^ set_name ~name c) sorted))

## Computing the Minimized <span style="font-variant:small-caps;">Dfa</span>

The function `minimize f` takes a <span style="font-variant:small-caps;">Dfa</span> `f` as its input.
Here `f` is a record that represents the 5-tuple
$$ F = (Q, \Sigma, \delta, q_0, A). $$
The <span style="font-variant:small-caps;">Dfa</span> $F$ has to be *complete*, i.e. $\delta(q, c)$ has to be defined for all states $q$ and all
characters $c$.  After removing the unreachable states, the function computes the separation table.
Then it returns the <span style="font-variant:small-caps;">Dfa</span>
$$ \texttt{Min}(F) := \bigl( \mathcal{Q}_\sim, \Sigma, \Delta, [q_0]_\sim, \widehat{A} \bigr), $$
that is defined in the lecture notes:
- $\mathcal{Q}_\sim := \bigl\{ [q]_\sim \bigm| q \in Q \bigr\}$ is the set of all equivalence classes,
- $\Delta\bigl([q]_\sim, c\bigr) := \bigl[\delta(q, c)\bigr]_\sim$ for all $q \in Q$ and $c \in \Sigma$, and
- $\widehat{A} := \bigl\{ [q]_\sim \bigm| q \in A \bigr\}$ is the set of accepting equivalence classes.

The function $\Delta$ is well defined:  If $p \sim q$, then we have $\delta(p, c) \sim \delta(q, c)$ for all
characters $c$, since otherwise $p$ and $q$ would be separable.

**Function `minimize f`**
- *Input:* `f` is a complete <span style="font-variant:small-caps;">Dfa</span>.
- *Output:* A <span style="font-variant:small-caps;">Dfa</span> with the minimal number of states that accepts the same language as `f`.  Its states are sets of equivalent states of `f`.

In [ ]:
let minimize (f : 'q dfa) : 'q min_dfa =
  let ({ sigma; delta; start = q0; accepting = a; _ } : 'q dfa) = f in
  let q            = reachable q0 sigma delta in
  let table        = separation_table q (FSet.inter a q) sigma delta in
  let equivClasses = FSet.map (fun p -> equivalence_class p q table) q in
  let newDelta : 'q trans_rel_set ref = ref Dict.empty in
  FSet.iter
    (fun p ->
      FSet.iter
        (fun c ->
          newDelta := Dict.add (equivalence_class p q table, c) (equivalence_class (Dict.find (p, c) delta) q table) !newDelta)
        sigma)
    q;
  let newQ0     = equivalence_class q0 q table in
  let newAccept = FSet.map (fun p -> equivalence_class p q table) (FSet.inter a q) in
  { states = equivClasses; sigma; delta = !newDelta; start = newQ0; accepting = newAccept }

## Drawing a <span style="font-variant:small-caps;">Dfa</span>

In [ ]:
load_lib "graphviz.ml"

The function `draw_dfa ~name f` draws the <span style="font-variant:small-caps;">Dfa</span> `f` as a graph.  In contrast to the function `dfa2dot` from the
notebook `FSM-2-Dot.ipynb`, the states are labelled with their names, so that they can be compared with the
tables.  If there are several characters that lead from a state $p$ to a state $q$, the graph contains only one
edge from $p$ to $q$ that is labelled with all of these characters.  The graph is displayed with `Graphviz.display`.

**Function `draw_dfa ~name f`**
- *Input:* `f` is a <span style="font-variant:small-caps;">Dfa</span>.
- *Output:* A graph of type `Graphviz.t` that shows `f`.

In [ ]:
let draw_dfa ~(name : 'q -> string) (f : 'q dfa) : Graphviz.t =
  let ({ states = q; sigma; delta; start = q0; accepting = a } : 'q dfa) = f in
  let dot = Graphviz.digraph "" in
  Graphviz.graph_attr dot "rankdir" "LR";
  let sorted = List.sort (fun p1 p2 -> compare (name p1) (name p2)) (FSet.to_list q) in
  let names  = Dict.of_list (List.mapi (fun n q -> (q, Printf.sprintf "S%d" n)) sorted) in
  let nm q = Dict.find q names in
  Graphviz.node dot ~attrs:[ ("label", ""); ("width", "0.1"); ("height", "0.1"); ("style", "filled"); ("color", "blue") ] "start";
  Graphviz.edge dot "start" (nm q0);
  FSet.iter
    (fun q ->
      if FSet.mem q a then Graphviz.node dot ~attrs:[ ("label", name q); ("peripheries", "2") ] (nm q)
      else Graphviz.node dot ~attrs:[ ("label", name q) ] (nm q))
    q;
  FSet.iter
    (fun p ->
      FSet.iter
        (fun q' ->
          let chars = List.filter (fun c -> Dict.find_opt (p, c) delta = Some q') (FSet.to_list sigma) in
          if chars <> [] then
            Graphviz.edge dot ~attrs:[ ("label", String.concat "," (List.map (String.make 1) chars)) ] (nm p) (nm q'))
        q)
    q;
  dot

## Examples

### The Example from the Lecture Notes

The <span style="font-variant:small-caps;">Dfa</span> `f1` is the <span style="font-variant:small-caps;">Dfa</span> that is used in the lecture notes in order to explain the algorithm.  The tables
computed in the steps 1 and 2 below coincide with the tables shown in the lecture notes.

In [ ]:
let delta1 : string trans_rel_det = Dict.of_list [
    (("q0", 'a'), "q1"); (("q0", 'b'), "q2");
    (("q1", 'a'), "q3"); (("q1", 'b'), "q3");
    (("q2", 'a'), "q4"); (("q2", 'b'), "q4");
    (("q3", 'a'), "q1"); (("q3", 'b'), "q1");
    (("q4", 'a'), "q2"); (("q4", 'b'), "q2");
  ]

let f1 : string dfa = { states = FSet.of_list [ "q0"; "q1"; "q2"; "q3"; "q4" ]; sigma = FSet.of_list [ 'a'; 'b' ];
                        delta = delta1; start = "q0"; accepting = FSet.of_list [ "q3"; "q4" ] }

In [ ]:
Graphviz.display (draw_dfa ~name:state_name f1)

In [ ]:
show_tables ~name:state_name f1

In [ ]:
let m1 = minimize f1
;;
Graphviz.display (draw_dfa ~name:(set_name ~name:state_name) m1)

### A More Complicated Example

The <span style="font-variant:small-caps;">Dfa</span> `f2` accepts all strings over the alphabet $\{\texttt{a}, \texttt{b}\}$ that end with the string
$\texttt{abb}$, i.e. we have
$$ L(F_2) = L\bigl((\texttt{a} + \texttt{b})^* \cdot \texttt{a} \cdot \texttt{b} \cdot \texttt{b}\bigr). $$
The states of `f2` are the strings of length $3$ over the alphabet $\{\texttt{a}, \texttt{b}\}$:  In the state
$xyz$ the <span style="font-variant:small-caps;">Dfa</span> remembers the last three characters $x$, $y$, and $z$ that it has read.  If it reads the character
$c$ in the state $xyz$, it switches into the state $yzc$.  The only accepting state is the state $\texttt{abb}$.
The start state is the state $\texttt{bbb}$, i.e. initially the <span style="font-variant:small-caps;">Dfa</span> acts as if it had already read the string
$\texttt{bbb}$.  This does not change the language that is accepted, as a string $w$ ends with $\texttt{abb}$ if
and only if the string $\texttt{bbb}w$ ends with $\texttt{abb}$.

In [ ]:
let q2 =
  let open FSet in
  let ab = of_list [ "a"; "b" ] in
  let* x = ab in
  let* y = ab in
  let* z = ab in
  return (x ^ y ^ z)

let delta2 : string trans_rel_det =
  Dict.of_list (List.concat_map (fun q -> List.map (fun c -> ((q, c), String.sub q 1 2 ^ String.make 1 c)) [ 'a'; 'b' ])
                                (FSet.to_list q2))

let f2 : string dfa = { states = q2; sigma = FSet.of_list [ 'a'; 'b' ]; delta = delta2; start = "bbb";
                        accepting = FSet.of_list [ "abb" ] }

In [ ]:
Graphviz.display (draw_dfa ~name:state_name f2)

For this <span style="font-variant:small-caps;">Dfa</span>, three steps find new pairs of separable states.

In [ ]:
show_tables ~name:state_name f2

In [ ]:
let m2 = minimize f2
;;
Graphviz.display (draw_dfa ~name:(set_name ~name:state_name) m2)

The minimized <span style="font-variant:small-caps;">Dfa</span> has only four states.  In order to decide whether a string ends with $\texttt{abb}$, the
<span style="font-variant:small-caps;">Dfa</span> only needs to remember the longest suffix of the input read so far that is also a prefix of
$\texttt{abb}$.  This suffix is either $\varepsilon$, $\texttt{a}$, $\texttt{ab}$, or $\texttt{abb}$:
- $\{\texttt{bbb}\}$ corresponds to the suffix $\varepsilon$,
- $\{\texttt{aaa}, \texttt{aba}, \texttt{baa}, \texttt{bba}\}$ contains the states that end with
  $\texttt{a}$ and corresponds to the suffix $\texttt{a}$,
- $\{\texttt{aab}, \texttt{bab}\}$ contains the states that end with $\texttt{ab}$ and corresponds to the
  suffix $\texttt{ab}$, and
- $\{\texttt{abb}\}$ corresponds to the suffix $\texttt{abb}$.